# 05. Profiling: Measure Before You Optimise

## 📚 Learning Objectives

By completing this notebook, you will:
- Take a **timing you can defend** — repeated, with a spread, using `time.perf_counter` and `%timeit`
- Read a **`cProfile`** report: `ncalls`, `tottime`, `cumtime`, and the difference that decides what you fix
- Find **where the time actually goes** in a small pandas pipeline, rather than where it looks like it goes
- Watch the **obvious optimisation fail** — it will be slower and use more memory than the code it "fixed"
- Measure **memory** with `df.memory_usage(deep=True)` and `tracemalloc`, and see why the default number under-reports
- Profile a **script from the command line** with `python -m cProfile`, the way you would on a repository you did not write

## 🔗 Where this fits

**Builds on:** Tooling strand, lessons 01-04. You need a shell, a working environment and a repository you can actually run before you can profile anything inside it. Profiling is usually the *first* thing you do to code somebody else wrote.

**Used later in:** Course 05 (AIAT 115) — Unit 5, lesson 06 "Performance Optimization", which opens by telling you to profile first and then choose between vectorising, compiling, distributing or moving to a GPU; Unit 1, lesson 08 "Introduction to Numba (JIT Compilation)"; and Unit 5, lesson 07 "Large Dataset Handling". This notebook is the measurement step those three assume you already have.

---

This is a **2-hour** lesson. Roughly half an hour is reading; the rest is you running measurements and arguing with them.

## 🎯 The claim this lesson tests

In December 1974, in *Structured Programming with go to Statements* (ACM Computing Surveys 6(4), 261-301), **Donald Knuth** wrote the most-quoted sentence in software performance: *"premature optimization is the root of all evil."* It is usually quoted as advice about tidiness. It is not. It is a claim about **evidence**: that programmers routinely spend effort speeding up code that was never the problem, because they guessed instead of measuring.

Fifty years later the guessing has not stopped, and the tooling built to stop it says so explicitly. **Scalene** (Berger, Stern & Altmayer Pizzorno, *Triangulating Python Performance Issues with Scalene*, OSDI '23, pages 51-64 — Best Paper) exists because Python programmers cannot tell, by reading, which of their lines is slow: its stated purpose is to help them **distinguish inefficient Python from efficient native execution**, because the two look identical in source code.

And Python's own standard-library documentation warns that the profiler itself is not neutral:

> "the profilers introduce overhead for Python code, but not for C-level functions, and so the C code would seem faster than any Python one."
> — *The Python Profilers*, Python 3 documentation

Three claims, then, and this notebook tests all three **on your machine, on real data, in the next hour**:

1. Your intuition about which line is slow is probably wrong.
2. The obvious optimisation is often not the one that matters — and can make things worse.
3. Even the profiler has a bias, and you have to know which way it leans.

Nothing below is a story about a company you cannot check. Every number in this notebook is produced by a cell you just ran.

## 📥 Inputs & 📤 Outputs

**Inputs:**
- `montgomery_911_calls` (the 25,000-row sample that ships with this repository) via the shared loader `tools.data`
- Standard library only: `time`, `cProfile`, `pstats`, `tracemalloc`, `subprocess`, `tempfile`
- No network, no model training, no GPU. Total runtime: a few seconds.

**Outputs:**
- Timings with a spread, not single numbers
- Two `cProfile` reports of the same pipeline, sorted two different ways
- A measured table of where the time went, against where a reader would guess it went
- Peak-memory figures for two implementations that return the identical result
- A real `python -m cProfile` run on a throwaway script in a scratch directory, which the notebook then deletes

---

In [1]:
# --- Data setup. Works from any folder, and on Google Colab. -------------------------
# WHAT: find the repository root and put it on sys.path, then import the shared loader.
# WHY:  a hard-coded '../../../Course 04/datasets/samples/...' only resolves when the
#       kernel's working directory happens to be this notebook's folder. This does not care.
import sys, pathlib

_here = pathlib.Path.cwd().resolve()
_root = next((p for p in [_here, *_here.parents] if (p / "tools" / "data.py").exists()), None)
if _root is None:                     # Google Colab, or a stray copy of the notebook
    import urllib.request
    pathlib.Path("tools").mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/A-Alwabel/"
            "AI-Diploma-Program/main/tools/data.py", "tools/data.py")
    except Exception as _e:
        raise RuntimeError(
            "Could not find the AI Diploma repository from this folder, and could not "
            "download the data loader either. Open this notebook inside a clone of "
            "https://github.com/A-Alwabel/AI-Diploma-Program, or connect to the internet "
            f"and re-run this cell. (underlying error: {_e})") from None
    _root = pathlib.Path.cwd()
sys.path.insert(0, str(_root))

from tools import data              # data.load(...), data.path(...), data.note(...)
# -------------------------------------------------------------------------------------

In [2]:
# WHAT: load the real Montgomery County 911 call log — every row is one emergency call.
# WHY:  prefer="sample" forces the 25,000-row copy that ships in the repository, so every
#       student profiles the SAME data and the ranking of the slow parts is comparable.
#       Timings will still differ between machines; the ranking is what we are studying.
import pandas as pd
import time

df = data.load("montgomery_911_calls", prefer="sample")

print(f"\nrows: {len(df):,}   columns: {len(df.columns)}")
print(f"townships (twp): {df['twp'].nunique()}   distinct call titles: {df['title'].nunique()}")
print("\nFirst three calls:")
print(df[["timeStamp", "twp", "title"]].head(3).to_string(index=False))
print("\n💡 Real dispatch records. Nobody chose these values; a county call centre logged them.")

montgomery_911_calls: bundled 25,000-row sample of the 663,522-row original (the full file is on this machine but was not used, because prefer='sample') — every number below is for the sample, not the full file. How it was drawn: 1 row in every 27, evenly spread, so the sample covers the same 2015-12-10 to 2020-07-29 window with all 100 call types and 68 townships.

rows: 25,000   columns: 9
townships (twp): 68   distinct call titles: 100

First three calls:
          timeStamp          twp                       title
2015-12-10 18:02:38     WHITPAIN            EMS: HEAD INJURY
2015-12-10 19:08:43     LANSDALE Traffic: ROAD OBSTRUCTION -
2015-12-10 20:36:49 LOWER MERION Traffic: VEHICLE ACCIDENT -

💡 Real dispatch records. Nobody chose these values; a county call centre logged them.


## ⏱️ Act 1 — one timing is not a measurement

The first mistake is not choosing the wrong optimisation. It is **believing a single stopwatch reading**.

A modern computer is doing hundreds of things while your code runs: other processes, the operating system's scheduler, CPU frequency scaling, caches that are cold on the first call and warm afterwards. Run the same function twice and you get two different numbers. So a timing is not a property of your code — it is a property of *your code on this machine, right now*.

Two rules follow, and every honest benchmark in the world obeys them:

1. **Repeat.** Never report one run.
2. **Report the spread, not just the middle.** If the fastest and slowest runs differ by 40%, any "18% improvement" you claim is noise.

Use `time.perf_counter()`, not `time.time()`. `perf_counter` is a monotonic high-resolution counter meant for measuring intervals; `time.time()` is the wall clock, and the wall clock can be adjusted underneath you.

In [3]:
# WHAT: time the same function once, then fifteen times, and compare what the two tell you.
# WHY:  this is the cell that should make you distrust every single-run benchmark you have
#       ever seen — including the ones you have written.

def label_calls(d):
    """Take the category out of each call title: 'EMS: HEAD INJURY' -> 'EMS'."""
    out = []
    for t in d["title"]:                 # a plain Python loop over 25,000 strings
        out.append(t.split(":")[0].strip())
    return out

# --- the naive way: one run, one number -------------------------------------------
t0 = time.perf_counter()
label_calls(df)
single = (time.perf_counter() - t0) * 1000
print(f"ONE run:            {single:8.3f} ms      <- the number most people report")

# --- the honest way: repeat, then look at the shape of the numbers -----------------
samples = []
for _ in range(15):
    t0 = time.perf_counter()
    label_calls(df)
    samples.append((time.perf_counter() - t0) * 1000)

samples_sorted = sorted(samples)
fastest, slowest = samples_sorted[0], samples_sorted[-1]
median = samples_sorted[len(samples_sorted) // 2]
spread = (slowest - fastest) / fastest * 100

print(f"15 runs: fastest    {fastest:8.3f} ms")
print(f"         median     {median:8.3f} ms")
print(f"         slowest    {slowest:8.3f} ms")
print(f"         spread     {spread:8.1f}%  (slowest is this much above fastest)")
print(f"\nThe single run above was slower than {sum(s < single for s in samples_sorted)} of the 15 repeated runs.")
print("\n💡 Report the FASTEST run when you compare implementations: it is the run least")
print("   polluted by other processes. Report the SPREAD so a reader can judge whether a")
print("   difference is real. A difference smaller than the spread is not a result.")

ONE run:               1.991 ms      <- the number most people report
15 runs: fastest       1.769 ms
         median        1.894 ms
         slowest       1.961 ms
         spread         10.8%  (slowest is this much above fastest)

The single run above was slower than 15 of the 15 repeated runs.

💡 Report the FASTEST run when you compare implementations: it is the run least
   polluted by other processes. Report the SPREAD so a reader can judge whether a
   difference is real. A difference smaller than the spread is not a result.


### 👁️ Read the numbers

Look at the spread your machine just printed. That percentage is your **noise floor**: any speed-up smaller than it is unmeasurable with this method, no matter how confident the person claiming it sounds.

This is also why the fastest run is the fair one to quote when *comparing two implementations*. Your code cannot run faster than its own best case; everything above that best case is interference from the rest of the machine. (When you want to predict how long something will take *in production*, the median or the 95th percentile is the honest number instead — the interference is real there too.)

In [4]:
# WHAT: the same measurement using IPython's %timeit, which does the repeating for you.
# WHY:  %timeit is what you will actually use day to day. -n = loops per run,
#       -r = number of runs, -o = return the result object so we can read the numbers.
timing = %timeit -o -q -n 5 -r 5 label_calls(df)

print("%timeit, unpacked:")
print(f"  best run   : {timing.best * 1000:8.3f} ms")
print(f"  worst run  : {timing.worst * 1000:8.3f} ms")
print(f"  mean        : {timing.average * 1000:8.3f} ms")
print(f"  std dev     : {timing.stdev * 1000:8.3f} ms")
print(f"  loops/run   : {timing.loops}    runs: {timing.repeat}")
print("\n💡 %timeit also disables the garbage collector while it runs. That makes runs")
print("   comparable, but it means the number is slightly optimistic versus real life.")
print("   Every benchmarking tool trades realism for repeatability somewhere.")

%timeit, unpacked:
  best run   :    1.801 ms
  worst run  :    1.896 ms
  mean        :    1.847 ms
  std dev     :    0.036 ms
  loops/run   : 5    runs: 5

💡 %timeit also disables the garbage collector while it runs. That makes runs
   comparable, but it means the number is slightly optimistic versus real life.
   Every benchmarking tool trades realism for repeatability somewhere.


## 🔬 Act 2 — a pipeline, and a prediction you write down first

Below is a small analysis of the call log. It answers three real questions: what kinds of calls come in, which hour is busiest, and how many calls each township makes. It is written the way working code is usually written — fast enough to ship, never measured.

Three stages:

| stage | what it does | how it looks |
|---|---|---|
| `label_calls` | pulls `EMS` / `Fire` / `Traffic` out of 25,000 title strings | **a plain Python `for` loop over 25,000 rows** |
| `busiest_hour` | parses 25,000 timestamps and counts calls per hour | one vectorised pandas line |
| `calls_per_township` | counts calls for each of the 68 townships | a `for` loop over **68** townships |

Before you run anything, commit to a guess. This matters: predicting and then being corrected is what changes your intuition. Reading the answer does not.

In [5]:
# WHAT: the three-stage pipeline, plus the real answers it produces.
# WHY:  everything after this cell is about WHERE this function spends its time. Notice
#       that it is not stupid code — it is ordinary code, and it gives correct answers.

def busiest_hour(d):
    """Which hour of the day receives the most 911 calls?"""
    stamps = pd.to_datetime(d["timeStamp"], format="%Y-%m-%d %H:%M:%S")
    return stamps.dt.hour.value_counts().idxmax()

def calls_per_township(d):
    """How many calls did each township make? One pass per township."""
    counts = {}
    for township in d["twp"].dropna().unique():
        counts[township] = len(d[d["twp"] == township])     # filter the whole frame, per township
    return counts

def report(d):
    """The whole pipeline: three stages, one call."""
    labels = label_calls(d)
    hour = busiest_hour(d)
    per_twp = calls_per_township(d)
    return labels, hour, per_twp

labels, hour, per_twp = report(df)

top_categories = pd.Series(labels).value_counts()
top_township = max(per_twp, key=per_twp.get)

print("THE ACTUAL ANSWERS (this is a real analysis, not a toy)")
print("=" * 62)
print("\nCalls by category:")
print(top_categories.to_string())
print(f"\nBusiest hour of the day : {hour}:00 - {hour}:59")
print(f"Busiest township        : {top_township} ({per_twp[top_township]:,} calls)")
print(f"Townships counted       : {len(per_twp)}")

THE ACTUAL ANSWERS (this is a real analysis, not a toy)

Calls by category:
EMS        12613
Traffic     8678
Fire        3709

Busiest hour of the day : 17:00 - 17:59
Busiest township        : LOWER MERION (2,102 calls)
Townships counted       : 68


### ✍️ Write your prediction down

In the next cell, set `my_prediction` to the stage you believe dominates the runtime, **before** you profile. Options:

- `"label_calls"` — the plain Python loop over 25,000 rows
- `"busiest_hour"` — parsing 25,000 timestamp strings
- `"calls_per_township"` — the loop over 68 townships

Most people pick the first, because "never loop over a DataFrame" is the most repeated advice in pandas. Pick whichever you actually believe, then run the cell.

In [6]:
# WHAT: your prediction, recorded before any measurement exists to contaminate it.
# WHY:  a prediction you wrote down is falsifiable. A feeling you had is not. Change the
#       string below to your own answer and re-run — the notebook checks it later.
my_prediction = "label_calls"        # <-- CHANGE ME before running the profiler

assert my_prediction in {"label_calls", "busiest_hour", "calls_per_township"}
print(f"Prediction recorded: {my_prediction}")
print("Now profile, and see whether the machine agrees with you.")

Prediction recorded: label_calls
Now profile, and see whether the machine agrees with you.


In [7]:
# WHAT: profile the pipeline with cProfile and print the report sorted two different ways.
# WHY:  cProfile records EVERY function call and how long it took. The two sort orders
#       answer two different questions, and mixing them up is the most common way to
#       misread a profile:
#         cumtime = time in this function AND everything it called  -> "which STAGE is heavy?"
#         tottime = time in this function's own body only           -> "which LINE is heavy?"
import cProfile, pstats, io

profiler = cProfile.Profile()
profiler.enable()
report(df)
profiler.disable()

buf = io.StringIO()
stats = pstats.Stats(profiler, stream=buf)
stats.sort_stats("cumtime").print_stats(10)
print("SORTED BY cumtime — which stage owns the time?")
print("=" * 62)
print(buf.getvalue())

buf = io.StringIO()
stats = pstats.Stats(profiler, stream=buf)
stats.sort_stats("tottime").print_stats(6)
print("SORTED BY tottime — which single piece of code is actually burning it?")
print("=" * 62)
print(buf.getvalue())

# print_stats() also takes a filter: only rows whose filename matches this pattern.
# Inside a notebook, the code YOU typed lives in files named .../ipykernel_*/....py
buf = io.StringIO()
stats = pstats.Stats(profiler, stream=buf)
stats.sort_stats("cumtime").print_stats("ipykernel", 6)
print("FILTERED to the code you typed in this notebook")
print("=" * 62)
print(buf.getvalue())

SORTED BY cumtime — which stage owns the time?
         114400 function calls (111749 primitive calls) in 0.060 seconds

   Ordered by: cumulative time
   List reduced from 449 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      3/2    0.000    0.000    0.052    0.026 /Users/abdullah/Downloads/AI Diploma/.venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3711(run_code)
      3/2    0.005    0.002    0.052    0.026 {built-in method builtins.exec}
        1    0.000    0.000    0.047    0.047 /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/ipykernel_41860/1974345585.py:1(<module>)
        1    0.000    0.000    0.047    0.047 /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/ipykernel_41860/1804176528.py:17(report)
        1    0.000    0.000    0.047    0.047 /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/ipykernel_41860/1804176528.py:10(calls_per_township)
       68    0.000    0.000    0.033    0.000 /Users/ab

### 👁️ Read the profile

Four columns matter.

- **`ncalls`** — how many times that function ran. A cheap function called a million times is the classic hidden cost.
- **`tottime`** — seconds spent *inside that function's own body*, excluding anything it called.
- **`cumtime`** — seconds spent inside it *and everything it called*, from entry to exit.
- **`filename:lineno(function)`** — where it lives. Lines with a `pandas/core/...` path are not your code; they are what your code asked pandas to do.

**First, learn to skip the noise.** The `cumtime` report does not open with your pipeline. Its top rows are `IPython/core/interactiveshell.py(run_code)` and `builtins.exec` — the notebook's own machinery for running a cell, which was inside the profiler when you switched it on. Scan down past those to the first row that is *your* code: the first is a temporary `ipykernel_.../....py(<module>)` file, which is the profiling cell itself, and just beneath it `...py(report)`, which is what a function defined in a notebook cell looks like to the profiler. Directly beneath `report` sits the stage that owns the pipeline. That is the row you were looking for — compare it against `my_prediction` above. The third block in the output does this filtering for you: `print_stats("ipykernel", 6)` keeps only rows whose filename matches that pattern, which in a notebook means the code you typed.

**Then read the `tottime` list**, and notice something uncomfortable: the heaviest single entry is **not a function you wrote**. It is `comp_method_OBJECT_ARRAY` inside `pandas/core/ops/array_ops.py` — the routine that compares an array of Python strings element by element. It is called 68 times, once per township, and each call compares all 25,000 rows. That is **68 × 25,000 = 1,700,000 string comparisons**, and it is the entire cost of a loop that looks harmless because it only runs 68 times.

**The size of a loop is not the number of iterations. It is iterations × the work inside.** That sentence is the whole lesson, and the profile is what makes it visible.

In [8]:
# WHAT: measure each stage on the wall clock (best of 5), then compare that split against
#       the split cProfile reported.
# WHY:  two reasons. First, to compute how much a "fix" to each stage could possibly buy
#       you. Second, because the profiler and the wall clock DISAGREE, and the direction
#       of the disagreement is documented behaviour you need to know about.

def best_ms(fn, repeats=5):
    """Fastest of several runs, in milliseconds. See Act 1 for why 'fastest'."""
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t0)
    return best * 1000

wall = {
    "label_calls":        best_ms(lambda: label_calls(df)),
    "busiest_hour":       best_ms(lambda: busiest_hour(df)),
    "calls_per_township": best_ms(lambda: calls_per_township(df)),
}
total_wall = sum(wall.values())

# The same three stages as cProfile saw them (cumulative seconds per stage).
prof = {}
for (fname, line, func), st in pstats.Stats(profiler).stats.items():
    if func in wall:
        prof[func] = st[3] * 1000          # st[3] is cumulative time, in seconds
total_prof = sum(prof.values())

print(f"{'stage':22s} {'wall ms':>9s} {'wall %':>8s} {'profiler %':>12s}   ceiling if made instant")
print("-" * 82)
for name in ("label_calls", "busiest_hour", "calls_per_township"):
    w_pct = wall[name] / total_wall * 100
    p_pct = prof[name] / total_prof * 100
    print(f"{name:22s} {wall[name]:9.2f} {w_pct:7.1f}% {p_pct:11.1f}%   at most {w_pct:5.1f}% faster overall")
print("-" * 82)
print(f"{'TOTAL':22s} {total_wall:9.2f} {100.0:7.1f}% {100.0:11.1f}%")

# How differently does each tool apportion the blame? 1.0 = the two agree.
distortion = {n: (prof[n] / total_prof) / (wall[n] / total_wall) for n in wall}
worst = max(distortion, key=lambda n: abs(distortion[n] - 1))
print(f"\nMOST DISTORTED STAGE: the profiler charges '{worst}' "
      f"{distortion[worst]:.1f}x its wall-clock share "
      f"({prof[worst] / total_prof * 100:.1f}% against {wall[worst] / total_wall * 100:.1f}%).")

dominant = max(wall, key=wall.get)
cheapest = min(wall, key=wall.get)
print(f"\nMEASURED VERDICT: '{dominant}' dominates the wall clock "
      f"({wall[dominant] / total_wall * 100:.1f}% of it).")
print(f"Your prediction was '{my_prediction}' -> "
      f"{'CORRECT' if my_prediction == dominant else 'WRONG, and now you know why we measure'}.")
print(f"\nDeleting '{cheapest}' entirely — making it take zero time — would speed the whole")
print(f"pipeline up by at most {wall[cheapest] / total_wall * 100:.1f}%. That is the ceiling on any")
print("effort you spend there. This is Amdahl's law in one line: the speed-up you can get")
print("from a part is capped by the fraction of time that part occupies.")

stage                    wall ms   wall %   profiler %   ceiling if made instant
----------------------------------------------------------------------------------
label_calls                 1.85     4.8%         5.7%   at most   4.8% faster overall
busiest_hour                1.87     4.8%        10.2%   at most   4.8% faster overall
calls_per_township         34.88    90.4%        84.1%   at most  90.4% faster overall
----------------------------------------------------------------------------------
TOTAL                      38.60   100.0%       100.0%

MOST DISTORTED STAGE: the profiler charges 'busiest_hour' 2.1x its wall-clock share (10.2% against 4.8%).

MEASURED VERDICT: 'calls_per_township' dominates the wall clock (90.4% of it).
Your prediction was 'label_calls' -> WRONG, and now you know why we measure.

Deleting 'label_calls' entirely — making it take zero time — would speed the whole
pipeline up by at most 4.8%. That is the ceiling on any
effort you spend there. This is A

### ⚠️ The profiler and the clock disagree — on purpose

Compare the `wall %` column with the `profiler %` column. They are not the same, and the cell named the stage where they disagree most.

That disagreement is not a bug and it is not random. `cProfile` is a **deterministic** profiler: it fires a hook on every function call and return. A stage whose work is spread across many small Python-level calls pays that hook once per call, and its share is inflated. A stage that does its work inside **one** C-level call — like the 1,700,000 string comparisons pandas performs inside a single `==` — pays the hook once in total, and its share shrinks. The Python documentation states the consequence plainly:

> "the profilers introduce overhead for Python code, but not for C-level functions, and so the C code would seem faster than any Python one."

So: **use `cProfile` to find *where* the time is, then confirm *how much* with a wall-clock timer.** A profile ranks; a stopwatch measures. If you only ever learn one thing about profiling, learn that these are two different jobs.

This is exactly the problem the Scalene paper (OSDI '23) was built to solve — separating inefficient Python from efficient native execution, which the standard profiler blurs together.

## 💥 Act 3 — the obvious optimisation, and why it fails

Everyone who reads `label_calls` reaches for the same fix: *stop looping, vectorise it.* Replace the `for` loop with pandas' string accessor:

```python
d["title"].str.split(":").str[0].str.strip()
```

One line instead of four. No visible loop. It looks like the textbook answer, and the advice behind it — "vectorise instead of looping" — is genuinely good advice most of the time.

Measure it anyway.

In [9]:
# WHAT: implement the obvious optimisation, check it returns the same answer, and time it.
# WHY:  an optimisation that changes the result is not an optimisation, it is a bug. Always
#       assert equality FIRST, then talk about speed.

def label_calls_vectorised(d):
    """The 'obvious' fix: pandas string accessor instead of a Python loop."""
    return d["title"].str.split(":").str[0].str.strip()

same = list(label_calls_vectorised(df)) == label_calls(df)
print(f"Same answer as the loop? {same}")
assert same, "an optimisation that changes the answer is a bug, not a speed-up"

loop_ms = best_ms(lambda: label_calls(df))
vec_ms = best_ms(lambda: label_calls_vectorised(df))

print(f"\nplain Python loop      : {loop_ms:8.3f} ms")
print(f"pandas .str accessor   : {vec_ms:8.3f} ms")

winner = "the pandas .str version" if vec_ms < loop_ms else "the plain Python loop"
factor = max(loop_ms, vec_ms) / min(loop_ms, vec_ms)
print(f"\nFASTER: {winner}, by {factor:.1f}x.")

# What does the 'fix' do to the pipeline as a whole?
def report_optimised(d):
    """Identical pipeline with the 'obvious' optimisation applied to stage one."""
    return label_calls_vectorised(d), busiest_hour(d), calls_per_township(d)

before = best_ms(lambda: report(df), repeats=3)
after = best_ms(lambda: report_optimised(df), repeats=3)
change = (after - before) / before * 100
print(f"\nWHOLE PIPELINE before : {before:8.2f} ms")
print(f"WHOLE PIPELINE after  : {after:8.2f} ms   ({change:+.1f}%)")
print("\n💡 Compare that percentage against the noise spread you measured in Act 1 before")
print("   calling it an improvement or a regression either way.")

Same answer as the loop? True

plain Python loop      :    1.912 ms
pandas .str accessor   :    4.673 ms

FASTER: the plain Python loop, by 2.4x.



WHOLE PIPELINE before :    39.36 ms
WHOLE PIPELINE after  :    42.20 ms   (+7.2%)

💡 Compare that percentage against the noise spread you measured in Act 1 before
   calling it an improvement or a regression either way.


### 👁️ Read the result

Two things just happened, and both are worth more than the milliseconds they cost.

**First, the "optimisation" did not win.** The pandas `.str` chain builds three intermediate Series — one from `split`, one from `[0]`, one from `strip` — and each element in them is still a Python string object, so pandas is looping in C over Python objects *and* allocating three whole columns on the way. The plain loop does one pass and appends to one list. Vectorisation buys you speed when the elements are **numbers** that NumPy can process in a tight typed loop. On object-dtype strings, that advantage largely evaporates, and the intermediates cost you.

**Second, and more important: it did not matter.** Whatever the sign of the change to the whole pipeline, it is a rounding error next to the stage you have not touched. You could have written the perfect implementation of `label_calls` — instantaneous, zero cost — and the ceiling on your reward was printed in the table above.

This is the shape of most wasted optimisation work in real projects: a genuinely clever change, correctly implemented, applied to the wrong function.

In [10]:
# WHAT: fix the stage the profiler actually accused, using one groupby instead of 68 filters.
# WHY:  the 68-iteration loop scanned all 25,000 rows on every iteration. groupby makes ONE
#       pass and buckets the rows as it goes. Same answer, different amount of work.

def calls_per_township_fast(d):
    """One pass over the data instead of one pass per township."""
    return d.groupby("twp").size().to_dict()

same = calls_per_township_fast(df) == calls_per_township(df)
print(f"Same answer as the 68-filter loop? {same}")
assert same, "an optimisation that changes the answer is a bug, not a speed-up"

slow_ms = best_ms(lambda: calls_per_township(df))
fast_ms = best_ms(lambda: calls_per_township_fast(df))
print(f"\n68 separate filters    : {slow_ms:8.3f} ms")
print(f"one groupby            : {fast_ms:8.3f} ms")
print(f"speed-up on the stage  : {slow_ms / fast_ms:8.1f}x")

def report_fixed(d):
    """The pipeline with the stage the PROFILER accused replaced — nothing else touched."""
    return label_calls(d), busiest_hour(d), calls_per_township_fast(d)

fixed = best_ms(lambda: report_fixed(df), repeats=3)
print(f"\nWHOLE PIPELINE original          : {before:8.2f} ms")
print(f"WHOLE PIPELINE 'obvious' fix     : {after:8.2f} ms   ({(after - before) / before * 100:+.1f}%)")
print(f"WHOLE PIPELINE profiler-led fix  : {fixed:8.2f} ms   ({(fixed - before) / before * 100:+.1f}%)")
print(f"\nThe profiler-led fix is {before / fixed:.1f}x faster overall. The obvious one changed "
      f"{abs((after - before) / before * 100):.1f}%.")
print("\n💡 Same programmer, same hour of work, same data. The only difference is that one")
print("   change was chosen by a measurement and the other was chosen by a habit.")

Same answer as the 68-filter loop? True



68 separate filters    :   35.047 ms
one groupby            :    0.483 ms
speed-up on the stage  :     72.5x

WHOLE PIPELINE original          :    39.36 ms
WHOLE PIPELINE 'obvious' fix     :    42.20 ms   (+7.2%)
WHOLE PIPELINE profiler-led fix  :     4.27 ms   (-89.1%)

The profiler-led fix is 9.2x faster overall. The obvious one changed 7.2%.

💡 Same programmer, same hour of work, same data. The only difference is that one
   change was chosen by a measurement and the other was chosen by a habit.


## 🧠 Act 4 — the memory question

Speed is the question people ask. Memory is the question that kills jobs. A slow pipeline finishes late; a pipeline that runs out of memory does not finish at all, and on a shared machine it can take other people's work down with it.

Two tools, two different questions:

- **`df.memory_usage(deep=True)`** — how much space is my data *sitting still* taking? (`deep=True` is not optional; see below.)
- **`tracemalloc`** — how much did my code *allocate while running*, and what was the peak? Peak is what decides whether you survive.

In [11]:
# WHAT: measure how much memory this DataFrame occupies, the wrong way and the right way.
# WHY:  memory_usage() without deep=True counts only the POINTERS in an object column,
#       not the Python strings they point at. It is the single most common way people
#       under-estimate a pandas frame — and the error is not small.
shallow_mb = df.memory_usage(deep=False).sum() / 1e6
deep_mb = df.memory_usage(deep=True).sum() / 1e6

print(f"df.memory_usage()               -> {shallow_mb:7.2f} MB   (pointers only: WRONG for text)")
print(f"df.memory_usage(deep=True)      -> {deep_mb:7.2f} MB   (the real figure)")
print(f"under-reported by                  {deep_mb / shallow_mb:7.2f}x\n")

per_col = (df.memory_usage(deep=True) / 1e6).sort_values(ascending=False)
print("Where the memory actually is, per column (MB):")
print(per_col.round(2).to_string())

# One cheap structural fix: columns with few distinct values become 'category'.
compact = df.copy()
for col in ["twp", "title"]:
    compact[col] = compact[col].astype("category")
compact_mb = compact.memory_usage(deep=True).sum() / 1e6
print(f"\ntwp ({df['twp'].nunique()} distinct) and title ({df['title'].nunique()} distinct) as 'category':")
print(f"  {deep_mb:.2f} MB -> {compact_mb:.2f} MB  ({(1 - compact_mb / deep_mb) * 100:.1f}% smaller)")
print("\n💡 A column with 25,000 rows and 68 distinct values does not need 25,000 strings.")
print("   'category' stores the 68 once and keeps small integer codes per row.")

df.memory_usage()               ->    1.80 MB   (pointers only: WRONG for text)
df.memory_usage(deep=True)      ->   10.68 MB   (the real figure)
under-reported by                     5.93x

Where the memory actually is, per column (MB):
desc         3.02
addr         1.86
title        1.79
timeStamp    1.70
twp          1.50
lat          0.20
lng          0.20
zip          0.20
e            0.20
Index        0.00



twp (68 distinct) and title (100 distinct) as 'category':
  10.68 MB -> 7.45 MB  (30.2% smaller)

💡 A column with 25,000 rows and 68 distinct values does not need 25,000 strings.
   'category' stores the 68 once and keeps small integer codes per row.


In [12]:
# WHAT: measure PEAK memory during each implementation with tracemalloc.
# WHY:  the two implementations of stage one return an identical answer. They do not cost
#       the same to run — and the difference is invisible in the source code.
import tracemalloc

def peak_mb(fn):
    """Peak Python-allocated memory during one call, in MB."""
    tracemalloc.start()
    fn()
    _current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / 1e6

loop_peak = peak_mb(lambda: label_calls(df))
vec_peak = peak_mb(lambda: label_calls_vectorised(df))
filter_peak = peak_mb(lambda: calls_per_township(df))
groupby_peak = peak_mb(lambda: calls_per_township_fast(df))

print(f"{'implementation':34s} {'peak MB':>9s}")
print("-" * 45)
print(f"{'label_calls (Python loop)':34s} {loop_peak:9.2f}")
print(f"{'label_calls (.str accessor)':34s} {vec_peak:9.2f}")
print(f"{'calls_per_township (68 filters)':34s} {filter_peak:9.2f}")
print(f"{'calls_per_township (groupby)':34s} {groupby_peak:9.2f}")
print("-" * 45)
print(f"\nThe 'obvious optimisation' peaks {vec_peak / loop_peak:.1f}x higher than the loop it replaced,")
print("because every intermediate Series in that .str chain is a full column held in memory")
print("at the same time. Scale the input up and that is the factor that decides whether the")
print("job finishes at all.")
print("\n⚠️ tracemalloc only sees memory allocated through Python's allocator. Buffers that a")
print("   C extension allocates directly are invisible to it, so treat these figures as a")
print("   LOWER BOUND on what the process actually used.")

implementation                       peak MB
---------------------------------------------
label_calls (Python loop)               1.36
label_calls (.str accessor)             8.07
calls_per_township (68 filters)         1.13
calls_per_township (groupby)            0.73
---------------------------------------------

The 'obvious optimisation' peaks 5.9x higher than the loop it replaced,
because every intermediate Series in that .str chain is a full column held in memory
at the same time. Scale the input up and that is the factor that decides whether the
job finishes at all.

⚠️ tracemalloc only sees memory allocated through Python's allocator. Buffers that a
   C extension allocates directly are invisible to it, so treat these figures as a
   LOWER BOUND on what the process actually used.


## 🖥️ Act 5 — profiling something that is not a notebook

You will be handed repositories, not notebooks. The tool for that is the same profiler, run from the shell:

```bash
python -m cProfile -s tottime your_script.py       # sort by own-body time
python -m cProfile -s cumtime your_script.py       # sort by total time including callees
python -m cProfile -o profile.out your_script.py   # save it, read it later with pstats
```

The cell below writes a real script into a **scratch directory that it creates and then deletes**, runs the real command against it with `subprocess`, and prints the real output. Nothing is written inside this repository.

In [13]:
# WHAT: write a throwaway script to a temporary directory, profile it from the command line
#       exactly as you would in a terminal, print the real output, then delete the directory.
# WHY:  because "python -m cProfile script.py" is the command you will actually type on a
#       repository somebody hands you, and because a lesson that only DESCRIBES a command
#       has not taught you to run it.
import subprocess, tempfile, shutil, textwrap
from pathlib import Path

scratch = Path(tempfile.mkdtemp(prefix="profiling_lesson_"))   # created outside the repo
print(f"scratch directory: {scratch}")

script = scratch / "township_report.py"
script.write_text(textwrap.dedent("""
    import sys
    import pandas as pd

    def calls_per_township(d):
        counts = {}
        for township in d["twp"].dropna().unique():
            counts[township] = len(d[d["twp"] == township])
        return counts

    def main(csv_path):
        df = pd.read_csv(csv_path, low_memory=False)
        counts = calls_per_township(df)
        print(f"{len(counts)} townships, busiest = {max(counts, key=counts.get)}")

    main(sys.argv[1])
"""))

csv_path = data.path("montgomery_911_calls", prefer="sample", quiet=True)
completed = subprocess.run(
    [sys.executable, "-m", "cProfile", "-s", "tottime", str(script), str(csv_path)],
    capture_output=True, text=True, timeout=180,
)
print(f"exit code: {completed.returncode}\n")
print("--- real output of: python -m cProfile -s tottime township_report.py <csv> ---")
print("\n".join(completed.stdout.splitlines()[:16]))

shutil.rmtree(scratch, ignore_errors=True)
print(f"\nscratch directory removed: {not scratch.exists()}")

scratch directory: /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/profiling_lesson_oeb3itkf


exit code: 0

--- real output of: python -m cProfile -s tottime township_report.py <csv> ---
68 townships, busiest = LOWER MERION
         535356 function calls (521719 primitive calls) in 0.284 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
       83    0.063    0.001    0.063    0.001 {built-in method _imp.create_dynamic}
       68    0.027    0.000    0.027    0.000 array_ops.py:113(comp_method_OBJECT_ARRAY)
      443    0.019    0.000    0.019    0.000 {built-in method marshal.loads}
        1    0.018    0.018    0.019    0.019 c_parser_wrapper.py:222(read)
    83/43    0.011    0.000    0.052    0.001 {built-in method _imp.exec_dynamic}
      443    0.006    0.000    0.006    0.000 {built-in method _io.open_code}
  956/945    0.006    0.000    0.026    0.000 {built-in method builtins.__build_class__}
     1626    0.005    0.000    0.010    0.000 textwrap.py:416(dedent)
     2160    0.004    0.000    0.004    0.000 {b

### 👁️ Read the command-line profile

Look at what sits at the top of that `tottime` list. The single largest entry is `_imp.create_dynamic` — loading pandas' compiled extension modules — with `marshal.loads` (unpacking `.pyc` files) and `c_parser_wrapper.py:read` (actually reading the CSV) close behind. The township loop's `comp_method_OBJECT_ARRAY` is there, and it is real, but on a run this short it is beaten by the cost of **starting Python and importing pandas at all**.

That is a true fact about the *script*, and a misleading one about the *function*. Both readings matter, and you have to know which question you asked:

- Profiling the **process** tells you why the command takes as long as it does — and for short jobs, start-up and I/O often dominate everything you wrote.
- Profiling the **function**, as we did with `cProfile.Profile()` around `report(df)`, tells you which of your code is slow.

If you profile a whole script and conclude "my code is fine, it is all imports", you have answered a question about a five-second command, not about a pipeline that will one day run on 2.3 million rows.

## 🛠️ Your turn

Two exercises. Do them in order; the second one is graded by a check you cannot satisfy by guessing.

**Exercise 1 — profile something you did not write.** `mystery_report` below has four stages with deliberately meaningless names. Read the code, write down which stage you think dominates, then *measure*. Set `my_answer` and run the checker.

**Exercise 2 — fix what the measurement accuses.** `calls_per_hour_slow` counts calls per hour of the day the naive way. Rewrite it in `calls_per_hour_fast` so that it returns **exactly** the same dictionary and is measurably faster. The checker verifies the answer first and the speed second — in that order, always.

The starting version of `calls_per_hour_fast` is a copy of the slow one, so the cell runs. Until you change it, the checker will say so.

In [14]:
# WHAT: Exercise 1. Four stages, opaque names, one of them dominates. Find it by measuring.
# WHY:  this is the actual job — you will be handed code you did not write, with no comments
#       telling you where the time goes, and asked why it is slow.

def step_one(d):
    return d["zip"].fillna(0).astype("int64")

def step_two(d):
    return d.apply(lambda row: f"{row['twp']}|{row['title']}", axis=1)

def step_three(d):
    seen = set()
    for address in d["addr"]:
        seen.add(address.strip().upper())
    return seen

def step_four(d):
    return d.groupby("twp")["e"].sum()

def mystery_report(d):
    return step_one(d), step_two(d), step_three(d), step_four(d)

# ---- 1. Measure the stages yourself. Use best_ms() from earlier, or cProfile. ----------
#         (Write your own measuring code here before you look at anything else.)


# ---- 2. Then record your answer. ------------------------------------------------------
my_answer = "step_three"       # <-- CHANGE ME to the stage you measured as dominant

# ---- 3. The checker measures the truth at run time and tells you if you were right. ----
assert my_answer in {"step_one", "step_two", "step_three", "step_four"}
measured = {name: best_ms(lambda f=fn: f(df), repeats=3)
            for name, fn in [("step_one", step_one), ("step_two", step_two),
                             ("step_three", step_three), ("step_four", step_four)]}
truth = max(measured, key=measured.get)
total = sum(measured.values())

print(f"{'stage':12s} {'ms':>8s} {'share':>8s}")
for name, ms in sorted(measured.items(), key=lambda kv: -kv[1]):
    print(f"{name:12s} {ms:8.2f} {ms / total * 100:7.1f}%")
print(f"\nyour answer: {my_answer}   measured: {truth}   -> "
      f"{'CORRECT' if my_answer == truth else 'WRONG'}")
print(f"\nAsk yourself why. Which of these four LOOKS like a Python loop, and which one IS")
print(f"one? The answer is not the same in both cases, and that is the point of Act 3.")

stage              ms    share
step_two        46.94    94.9%
step_three       1.97     4.0%
step_four        0.48     1.0%
step_one         0.08     0.2%

your answer: step_three   measured: step_two   -> WRONG

Ask yourself why. Which of these four LOOKS like a Python loop, and which one IS
one? The answer is not the same in both cases, and that is the point of Act 3.


In [15]:
# WHAT: Exercise 2. Make calls_per_hour_fast return the same dictionary, faster.
# WHY:  correctness first, speed second — that order is not a style preference, it is the
#       difference between an optimisation and a silent data bug.

def calls_per_hour_slow(d):
    """Calls per hour of the day. Written the way tired people write things at 5pm."""
    counts = {}
    for h in range(24):
        hours = pd.to_datetime(d["timeStamp"], format="%Y-%m-%d %H:%M:%S").dt.hour
        counts[h] = len(d[hours == h])
    return counts

def calls_per_hour_fast(d):
    """YOUR CODE HERE. It currently just calls the slow version, so the notebook runs.
    Replace the body. Return a dict {hour: count} for every hour 0-23 that appears."""
    return calls_per_hour_slow(d)          # <-- replace this line with your implementation

# ---- checker: identical answer first, then speed -------------------------------------
expected = calls_per_hour_slow(df)
got = calls_per_hour_fast(df)

if got != expected:
    print("❌ WRONG ANSWER — your version does not match the slow one. Speed is irrelevant")
    print("   until this line says CORRECT. Compare a few hours by hand to find the gap.")
else:
    slow = best_ms(lambda: calls_per_hour_slow(df), repeats=3)
    fast = best_ms(lambda: calls_per_hour_fast(df), repeats=3)
    print(f"✅ correct answer   slow: {slow:.2f} ms   yours: {fast:.2f} ms   "
          f"({slow / fast:.2f}x)")
    if fast >= slow * 0.9:
        print("\nSTATUS: not improved yet — this is the starting state of the exercise.")
        print("Profile calls_per_hour_slow before you touch it. Ask the profile two things:")
        print("which line runs 24 times, and how much of what it does never changes between")
        print("those 24 runs. There are two separate wins in there. Find both.")
    else:
        print("\nSTATUS: improved. Now state your speed-up WITH the noise spread from Act 1,")
        print("the way you would have to defend it in a code review.")

✅ correct answer   slow: 43.55 ms   yours: 43.67 ms   (1.00x)

STATUS: not improved yet — this is the starting state of the exercise.
Profile calls_per_hour_slow before you touch it. Ask the profile two things:
which line runs 24 times, and how much of what it does never changes between
those 24 runs. There are two separate wins in there. Find both.


## 💬 Discuss

Everything you need is in the outputs above — your Act 1 spread, the wall-versus-profiler table, the three pipeline totals, and the peak-memory figures.

1. **The obvious optimisation changed the pipeline by a percentage you printed; the profiler-led one changed it by a much larger factor.** Now argue the other side: name a situation in a real project where you would take the small change and refuse the large one. (Hint: read `calls_per_township_fast` and ask what it does to the code someone else has to maintain, and what happens if the requirement changes to "counts per township *per month*".)

2. **Your `cProfile` percentage for `label_calls` was higher than its wall-clock percentage.** A colleague sends you a profile as evidence that a loop must be rewritten. What do you ask them for before you agree, and what would change your mind?

3. **This notebook profiles 25,000 rows. The full call log has 663,522, and `cicids2017` in this repository has 2.3 million.** For each of the three stages, say whether its cost grows *in proportion* to the rows or *faster than* the rows — and therefore which stage you should fix first if you knew the data was about to get 100 times bigger. Justify each answer from what you measured, not from the shape of the code.

4. **Saudi context:** pick a system you have actually seen — a government portal, a delivery app at Ramadan peak, a university registration morning. If it slowed down, whose job is it to produce the profile, and what would stop them? Name the organisational obstacle, not the technical one.

## Summary

This notebook covered:
- ✅ **Timing honestly**: `time.perf_counter`, repeated runs, fastest-plus-spread, and why one run is not a measurement
- ✅ **`%timeit`**: what `-n`, `-r` and `-o` mean, and what it hides (the garbage collector)
- ✅ **`cProfile` and `pstats`**: `ncalls`, `tottime`, `cumtime`, and which question each one answers
- ✅ **Where the time actually goes**: 68 iterations doing 1,700,000 string comparisons, versus 25,000 iterations doing almost nothing
- ✅ **The obvious optimisation failing**: vectorising object-dtype strings, correct and clever and beside the point
- ✅ **Amdahl's ceiling**: the speed-up available from a part is capped by that part's share of the time
- ✅ **Memory**: `memory_usage(deep=True)`, `category` dtype, and `tracemalloc` peaks that differ between two implementations of the same answer
- ✅ **`python -m cProfile`** on a real script in a scratch directory, and why the top of that report is usually `import pandas`

**The honest takeaway:** the skill is not knowing fast tricks. Anyone can memorise "use `groupby`, avoid `iterrows`". The skill is refusing to change a line until a measurement tells you which line — and then measuring again to prove that the change did what you claimed. In this notebook, the reflex answer was both slower and larger in memory than the code it was supposed to improve, and the profiler found the real cost in a loop that only ran 68 times.

## ⚠️ Where this breaks

- **This whole strand has no outcome evidence behind it.** There is no study showing that teaching students the shell, environments, version control or profiling raises their employment or performance outcomes. It is in this diploma because everything else the programme promises — peer review of real work, being handed a repository and being useful in it — is undeliverable without it. That is a **prerequisite argument, not a proven intervention**, and you should hear it as one. Compare that honestly with the peer-review strand, which does have measured effects.

- **A timing is a fact about a machine, not about code.** Everything you measured is true of this CPU, this pandas version, this Python build, with this much free memory, on battery or on mains. The ranking of the three stages is fairly robust; the exact factors are not. A profile taken on your laptop can select the wrong optimisation for the server.

- **`cProfile` systematically over-charges Python and under-charges C**, as the Python documentation states and as the wall-versus-profiler table above demonstrated. Use it to rank, not to quantify. Sampling profilers (`py-spy`, `Scalene`, `austin`) have the opposite trade-off: lower distortion, but they can miss short-lived functions entirely.

- **Micro-benchmarks do not extrapolate.** 25,000 rows tells you almost nothing about 2.3 million, because the stages have different growth rates and because at some size the data stops fitting in cache, then in memory, and the ranking changes completely.

- **`tracemalloc` sees only Python's allocator.** Memory that NumPy or another C extension takes directly from the operating system does not appear. The peaks above are a floor, not a total. For the real figure you need the process's resident set size (`psutil`), and for line-by-line memory a dedicated tool.

- **The first run is different from the rest.** Cold file caches, lazy imports and JIT warm-up all inflate run one. Everything above deliberately reports the *fastest* run, which hides exactly that cost — and in a service that starts fresh for every request, the cost you hid is the only one your users feel.

- **Optimisation has a price in readability, and the profile never shows it.** `calls_per_township_fast` is faster and it is also less obvious to a junior colleague at 2 a.m. Every speed-up is a trade against maintenance, and the person who pays is rarely the person who made the trade.

- **When not to bother:** if the whole job takes 40 milliseconds and runs once a day, the correct optimisation is none. Knuth's sentence is not about laziness — it is about spending your finite attention where a measurement says it will pay.

## 📚 References

1. Knuth, D. E. (1974). *Structured Programming with go to Statements*. ACM Computing Surveys, 6(4), 261-301. The source of "premature optimization is the root of all evil", and of the argument that the critical part must be identified by measurement rather than by intuition.
2. Python Software Foundation. *The Python Profilers — `profile` and `cProfile`*. Python 3 standard library documentation. <https://docs.python.org/3/library/profile.html> — definitions of `tottime` and `cumtime`, and the warning that the profilers add overhead to Python code but not to C-level functions.
3. Berger, E. D., Stern, S., & Altmayer Pizzorno, J. (2023). *Triangulating Python Performance Issues with Scalene*. 17th USENIX Symposium on Operating Systems Design and Implementation (OSDI '23), 51-64. Best Paper. <https://www.usenix.org/conference/osdi23/presentation/berger> (preprint: <https://arxiv.org/abs/2212.07597>) — a profiler built specifically to separate inefficient Python from efficient native execution, and to track memory and copy volume.
4. Amdahl, G. M. (1967). *Validity of the Single Processor Approach to Achieving Large Scale Computing Capabilities*. AFIPS Spring Joint Computer Conference, 483-485. The ceiling computed in the wall-clock table above.
5. Python Software Foundation. *`tracemalloc` — Trace memory allocations*. Python 3 standard library documentation.
6. Gorelick, M., & Ozsvald, I. (2020). *High Performance Python* (2nd ed.). O'Reilly Media. Chapters 2 and 11 cover profiling method and memory measurement in more depth than a single lesson can.